# 🎬 Battle of Colachel — Cloud GPU Blender Renderer

This Google Colab notebook offloads Blender rendering to a **free cloud NVIDIA GPU** (T4 / L4), keeping your local machine cool and speeding up rendering.

### Quick Setup Steps:
1. Go to **Runtime > Change runtime type** and ensure **T4 GPU** is selected.
2. Run **Step 1 & 2** to verify GPU and install headless Linux graphics libraries.
3. Run **Step 3** to automatically download **Blender 5.2.2 LTS** for Linux.
4. Run **Step 4** to load the sample scene (auto-detects uploaded zip/blend, or prompts to upload).
5. Run **Step 5 & 6** to render and preview a sample frame right in this notebook!
6. Run **Step 7 & 8** to render an animation sequence and download the `.zip` archive.

### Step 1: Verify GPU Allocation
Make sure an NVIDIA GPU is assigned to your Colab session.

In [ ]:
!nvidia-smi

### Step 2: Configure Cloud GPU & Headless Hardware Acceleration
Installs required Linux windowing/graphics runtime libraries (`libglvnd`, `libxi6`, `ffmpeg`) and configures NVIDIA EGL so EEVEE renders directly on the NVIDIA Cloud GPU without falling back to slow CPU software rasterization (llvmpipe).

In [ ]:
import os
# Install essential runtime libraries for headless cloud rendering
!apt-get update -qq
!apt-get install -y -qq libxi6 libxrender1 libxkbcommon-x11-0 libglvnd0 libgl1 libegl1 ffmpeg

# Point EGL directly to NVIDIA's hardware driver (prevents llvmpipe CPU fallback)
os.environ['__EGL_VENDOR_LIBRARY_FILENAMES'] = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
os.environ['__GLX_VENDOR_LIBRARY_NAME'] = 'nvidia'
if 'DISPLAY' in os.environ:
    del os.environ['DISPLAY']

print('✓ Cloud GPU hardware acceleration configured successfully!')

### Step 3: Download & Extract Blender 5.2.2 LTS
Downloads the official Linux binary matching your local project version.

In [ ]:
import os

BLENDER_VERSION = '5.2.2'
BLENDER_DIR = f'blender-{BLENDER_VERSION}-linux-x64'
BLENDER_TAR = f'{BLENDER_DIR}.tar.xz'
BLENDER_URL = f'https://download.blender.org/release/Blender5.2/{BLENDER_TAR}'

if not os.path.exists(BLENDER_DIR):
    print(f'Downloading Blender {BLENDER_VERSION} LTS...')
    !wget -q --show-progress {BLENDER_URL}
    print(f'Extracting {BLENDER_TAR}...')
    !tar -xf {BLENDER_TAR}
    print(f'✓ Blender {BLENDER_VERSION} installed successfully!')
else:
    print(f'✓ Blender {BLENDER_VERSION} already present.')

!./{BLENDER_DIR}/blender --version

### Step 4: Load Project Scene
This cell automatically finds any uploaded `.zip` or `.blend` file. If none is found yet, it prompts you with a file upload button!

In [ ]:
import os
import glob
import shutil
import zipfile
from google.colab import files

PROJECT_DIR = '/content/project'
OUTPUT_DIR = '/content/renders'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Function to find and unpack files
def find_and_setup_files():
    # Check for any uploaded zip in /content
    zips = glob.glob('/content/*.zip')
    for zpath in zips:
        try:
            with zipfile.ZipFile(zpath, 'r') as z:
                z.extractall(PROJECT_DIR)
            print(f'✓ Extracted {zpath} into {PROJECT_DIR}')
        except Exception as e:
            print(f'Could not extract {zpath}: {e}')

    # Check for any uploaded .blend in /content
    blends_in_content = glob.glob('/content/*.blend')
    for bpath in blends_in_content:
        shutil.copy(bpath, PROJECT_DIR)
        print(f'✓ Copied {bpath} into {PROJECT_DIR}')

    # Check for any colab_render_cli.py in /content
    if os.path.exists('/content/colab_render_cli.py'):
        shutil.copy('/content/colab_render_cli.py', PROJECT_DIR)

find_and_setup_files()

# Search inside PROJECT_DIR for a .blend file
project_blends = glob.glob(f'{PROJECT_DIR}/*.blend')

# If no .blend found, prompt user with interactive upload button
if not project_blends:
    print('👉 No .blend file detected in /content yet.')
    print('Please click "Choose Files" below to upload "sample_render_package.zip" or "sample_scene.blend":')
    uploaded = files.upload()
    find_and_setup_files()
    project_blends = glob.glob(f'{PROJECT_DIR}/*.blend')

if project_blends:
    BLEND_FILE = project_blends[0]
    print(f'\n✅ Ready! Found target blend: {BLEND_FILE}')
else:
    raise FileNotFoundError('❌ No .blend file found! Please upload sample_render_package.zip or sample_scene.blend and re-run this cell.')

SCRIPT_PATH = os.path.join(PROJECT_DIR, 'colab_render_cli.py')
if not os.path.exists(SCRIPT_PATH):
    # If CLI script was not in the package, write a minimal helper script
    with open(SCRIPT_PATH, 'w') as f:
        f.write('''import bpy, sys, os\nargs = sys.argv[sys.argv.index("--")+1:] if "--" in sys.argv else []\nmode = args[0] if len(args)>0 else "stills"\nout = args[1] if len(args)>1 else "/content/renders/out"\nsc = bpy.context.scene\nsc.render.resolution_x, sc.render.resolution_y = 1280, 720\nsc.render.resolution_percentage = 100\nsc.frame_set(1)\nsc.render.filepath = f"{out}_f0001.png"\nbpy.ops.render.render(write_still=True)\nprint("Saved:", sc.render.filepath)\n''')
    print(f'✓ Created helper script at {SCRIPT_PATH}')
else:
    print(f'✓ Using render script: {SCRIPT_PATH}')


### Step 5: Run Single-Frame Sample Render Test
Renders Frame 1 at 720p directly on the NVIDIA GPU via headless EGL (typical speed: ~2–5 seconds/frame instead of 2 minutes on CPU).

In [ ]:
# Verify file exists before launching Blender
import os
if not os.path.exists(BLEND_FILE):
    raise FileNotFoundError(f'Cannot find {BLEND_FILE}. Please re-run Step 4.')

print(f'Rendering test still from {BLEND_FILE} on Cloud GPU...')
# Direct headless render via NVIDIA GPU EGL (no xvfb CPU bottleneck)
!./{BLENDER_DIR}/blender -b "{BLEND_FILE}" \
  -P "{SCRIPT_PATH}" -- stills "{OUTPUT_DIR}/test_still" 1 --res 1280x720

### Step 6: Preview Rendered Frame Inline
Inspect the rendered frame directly inside the Colab notebook.

In [ ]:
from IPython.display import Image, display
import glob

stills = sorted(glob.glob(f'{OUTPUT_DIR}/test_still*.png'))
if stills:
    print(f'Displaying rendered output: {stills[-1]}')
    display(Image(filename=stills[-1], width=850))
else:
    print('No test still image found. Check Step 5 output for any errors.')

### Step 7: Batch Animation Render (Custom Frame Range)
Renders an animation sequence (e.g. frames 1 to 24) on the NVIDIA GPU. Includes automatic resume: skips any already-rendered valid frames.

In [ ]:
START_FRAME = 1
END_FRAME = 24  # Change this range to suit your shot
ANIM_DIR = f'{OUTPUT_DIR}/frames_sample_720p'

print(f'Rendering animation frames {START_FRAME}..{END_FRAME} to {ANIM_DIR} on Cloud GPU...')
!./{BLENDER_DIR}/blender -b "{BLEND_FILE}" \
  -P "{SCRIPT_PATH}" -- frames "{ANIM_DIR}" --range {START_FRAME} {END_FRAME} --res 1280x720

### Step 8: Package & Download Rendered Frames
Compresses rendered frames into a `.zip` archive and triggers an automatic browser download.

In [ ]:
import shutil
from google.colab import files

archive_name = '/content/rendered_frames'
shutil.make_archive(archive_name, 'zip', ANIM_DIR)
zip_file = f'{archive_name}.zip'
print(f'✓ Created {zip_file}. Starting browser download...')
files.download(zip_file)